In [25]:
import os
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr
from scipy import signal
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [26]:
class TemporalFeatures:

    def __init__(self, path_csv, path_odio, start, syllabs, cutoff, nyq_freq, padding):
        self.path_csv = path_csv
        self.path_odio = path_odio
        self.start = start
        self.syllabs = syllabs
        self.cutoff = cutoff
        self.nyq_freq = nyq_freq
        self.padding = padding

        self.df = pd.read_csv(path_csv)
        self.df = self.df[self.df.Syllables == syllabs]

        # Precompute filter coefficients once (optimization)
        self.b, self.a = signal.butter(4, cutoff / nyq_freq, btype="lowpass")

    def chirp_2_idx(self):
        """Map chirp to (start, end) indices"""
        return {chirp: (st, ed) for chirp, _, st, ed, _, _, _, _ in self.df.to_records(index=False)}

    def chirp_2_audio(self):
        """Map chirp to audio file"""
        return {chirp: audio for chirp, _, _, _, _, _, _, audio in self.df.to_records(index=False)}

    @staticmethod
    def audio_2_chirps(ch2odio_dict):
        """Group chirps by audio file"""
        audio2ch = {}
        for chirp, audio in ch2odio_dict.items():
            if audio not in audio2ch:
                audio2ch[audio] = []
            audio2ch[audio].append(chirp)
        return audio2ch

    def butter_lowpass_filter(self, data):
        """Apply lowpass filter using precomputed coefficients"""
        return signal.filtfilt(self.b, self.a, data)

    def call_signal(self, audiofile):
        """Load audio, reduce noise, and apply filtering"""
        signal_raw, sr = librosa.load(os.path.join(self.path_odio, audiofile), sr=None, offset=self.start)
        reduced_noise = nr.reduce_noise(y=signal_raw, sr=sr)
        filtered = self.butter_lowpass_filter(reduced_noise)
        return filtered, signal_raw, sr

    @staticmethod
    def detect_cricket_call(signal, spacing=235, padding=50):
        """Detect syllable start and end positions"""
        signal = signal / (np.abs(signal).max() + 1e-12)
        idx = np.where(np.abs(signal) > 0.1)[0]
        
        if len(idx) == 0:
            return np.array([]), np.array([])
            
        diff = np.diff(idx)
        starts = idx[np.hstack(([True], diff > spacing))]

        rev_signal = signal[::-1]
        idx_rev = np.where(np.abs(rev_signal) > 0.1)[0]
        diff_rev = np.diff(idx_rev)

        ends = rev_signal.shape[0] - idx_rev[np.hstack(([True], diff_rev > spacing))]

        starts = np.maximum(0, starts - padding)
        ends = np.minimum(ends[::-1] + padding, signal.shape[0])

        return starts, ends

    @staticmethod
    def carrier_frequency(signal, sr):
        """Estimate dominant carrier frequency"""
        D = librosa.stft(signal)
        freqs = librosa.fft_frequencies(sr=sr)
        return freqs[np.argmax(np.mean(np.abs(D), axis=1))]

    @staticmethod
    def mfcc_features(signal):
        """Extract MFCC mean and std features"""
        signal = signal / (np.abs(signal).max() + 1e-12)

        mfccs = librosa.feature.mfcc(
            y=signal, sr=44100,
            n_mfcc=20, n_mels=128,
            fmin=4000, fmax=8000,
            n_fft=1024, hop_length=256
        )

        return np.hstack([mfccs.mean(axis=1), mfccs.std(axis=1)])

    def temporal_feature_names(self):
        """Generate feature column names"""

        if self.syllabs == 5:
            tfs = ['chDur','s1Dur','s2Dur','s3Dur','s4Dur','s5Dur','meanSyl','s12Gap','s23Gap','s34Gap','s45Gap','meanGap','carrier_freq']
        elif self.syllabs == 4:
            tfs = ['chDur','s1Dur','s2Dur','s3Dur','s4Dur','meanSyl','s12Gap','s23Gap','s34Gap','meanGap','carrier_freq']
        elif self.syllabs == 3:
            tfs = ['chDur','s1Dur','s2Dur','s3Dur','meanSyl','s12Gap','s23Gap','meanGap','carrier_freq']
        else:
            tfs = ['chDur','s1Dur','s2Dur','meanSyl','s12Gap','meanGap','carrier_freq']

        mfcc_names = [f"x{i}" for i in range(40)]
        return ["audio"]+ mfcc_names + tfs

    def Execution(self):
        """Main feature extraction pipeline"""

        ch2idx = self.chirp_2_idx()
        ch2odio = self.chirp_2_audio()
        audio2chirps = self.audio_2_chirps(ch2odio)

        chfs = {}

        for audio, chirps in tqdm(audio2chirps.items()):

            filtered, signal_raw, sr = self.call_signal(audio)

            for chirp in chirps:

                st, ed = ch2idx[chirp]

                chirp_sig = signal_raw[st:ed]
                chirp_filtered = filtered[st:ed]

                starts, ends = self.detect_cricket_call(chirp_sig)

                if ends.shape[0] != self.syllabs:
                    continue

                chDurs = np.hstack([(ed-self.padding)-(st+self.padding), (ends-starts), (ends-starts).mean()])
                chGaps = np.hstack([(starts[1:]-ends[:-1]), (starts[1:]-ends[:-1]).mean()])

                mfcc_feats = self.mfcc_features(chirp_filtered)
                carrier_freq = self.carrier_frequency(chirp_filtered, sr)

                chfs[chirp] = np.hstack([audio, mfcc_feats, chDurs, chGaps, carrier_freq])

        df = pd.DataFrame.from_dict(chfs, orient='index', columns=self.temporal_feature_names())
        return df.rename_axis("chirp")


In [27]:
if __name__ == "__main__":

    path_chirp_data = "../chirp_index_extraction/chirp_data.csv"
    path_odio = "../audio_files/"
    start = 10
    syllabs = 5
    cutoff = 8000
    nyq_freq = 44100 / 2
    padding = 400

    tF = TemporalFeatures(path_chirp_data, path_odio, start, syllabs, cutoff, nyq_freq, padding)
    df = tF.Execution()
    df.to_csv("mfcc_tfs_carrier_freq_5syllable_chirps.csv", index = True)

100%|█████████████████████████████████████████████| 7/7 [00:11<00:00,  1.71s/it]


In [28]:
df

,audio,x0,x1,x2,x3,x4,x5,x6,x7,x8,...,s3Dur,s4Dur,s5Dur,meanSyl,s12Gap,s23Gap,s34Gap,s45Gap,meanGap,carrier_freq
chirp,,,,,,,,,,,,,,,,,,,,,
LPL11_0,LPL_1.WAV,-305.72396537419866,13.454308134196074,-190.88496337991933,-25.69739573916252,27.027141682903935,-13.906114433782227,-1.5178472367298443,6.408868935765031,6.859609448076383,...,802.0,905.0,924.0,759.2,177.0,220.0,818.0,920.0,533.75,5727.83203125
LPL11_2,LPL_1.WAV,-309.82637459452724,25.674900680544035,-181.9074019997997,-29.372532989282295,20.836538143548363,13.082485327981543,-15.81838882076354,6.1217752643968515,10.659213090969388,...,818.0,889.0,849.0,734.4,243.0,243.0,843.0,1006.0,583.75,5534.033203125
LPL11_6,LPL_1.WAV,-387.88291888692527,58.23864889248011,-196.73640181112899,-62.97328585201765,65.84103497597262,-18.34336203442975,-11.613263546122123,25.01488736348846,2.0434012596422058,...,816.0,896.0,913.0,746.0,194.0,236.0,806.0,1025.0,565.25,5620.166015625
LPL11_8,LPL_1.WAV,-357.2906258793788,34.705038571156884,-196.69799992923262,-36.462116570498374,36.92118490934295,-8.911637104073392,-1.9354416241816232,14.028460383414599,9.216344390835737,...,818.0,930.0,915.0,752.6,213.0,165.0,836.0,1044.0,564.5,5620.166015625
LPL11_9,LPL_1.WAV,-330.96739378935183,27.216075817533348,-188.69441672328958,-58.59325236991839,34.75072863864864,-13.857949240132685,-0.1458286528817502,13.633344442160412,7.040383827859196,...,784.0,869.0,902.0,748.4,214.0,252.0,808.0,1007.0,570.25,5534.033203125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RBB13_256,RBB_n3_1.WAV,-336.7570806207932,33.16761381688374,-193.59837513078338,-36.0955890115654,37.58052966983457,-5.719470273627968,-7.303418293762193,17.68236816044453,-13.70451410834187,...,1013.0,1076.0,1070.0,910.2,316.0,306.0,916.0,1069.0,651.75,5727.83203125
RBB13_258,RBB_n3_1.WAV,-342.6661326363085,34.066108115579475,-191.68601740833168,-31.10594252625658,39.76127264514968,-10.131181238003215,-8.197198616753921,12.42976606222959,-6.242005292533439,...,1032.0,1065.0,1082.0,923.6,339.0,310.0,959.0,1106.0,678.5,5684.765625
RBB13_259,RBB_n3_1.WAV,-349.2217536628745,41.44074344906876,-189.48031653452682,-26.954034752353763,44.552393332471944,1.3099184591306579,1.8016388402442034,7.439803514503186,-2.5397855644371523,...,1006.0,1064.0,1067.0,884.6,365.0,279.0,984.0,1126.0,688.5,5878.564453125


In [30]:
df["audio"].unique()

array(['LPL_1.WAV', 'BKB_2.WAV', 'RBB_n1_1.WAV', 'LPL_3.WAV',
       'BKB_n2_1.WAV', 'LPL_2.WAV', 'RBB_n3_1.WAV'], dtype=object)